# Lesson 04 - รูปแบบการออกแบบการใช้เครื่องมือ

ในบทเรียนนีี้คุณจะได้เรียนรู้รูปแบบการออกแบบ **การใช้เครื่องมือ** สำหรับเอเจนต์ AI โดยใช้ Microsoft Agent Framework (Python) เราจะครอบคลุมถึง:

- การกำหนดฟังก์ชันเครื่องมือด้วยการตกแต่ง `@tool` และพารามิเตอร์ที่มีการระบุประเภท
- การให้สคีมาของเครื่องมือเพื่อให้โมเดลรู้ว่าทุกเครื่องมือทำงานอย่างไร
- การควบคุมการใช้งานเครื่องมือด้วย `approval_mode`
- การส่งคืน **ผลลัพธ์ที่มีโครงสร้าง** ผ่านโมเดล Pydantic และ `response_format`

สถานการณ์คือ **เอเจนต์จองการเดินทาง** ที่สามารถค้นหาสถานที่ปลายทาง ตรวจสอบความพร้อมใช้งาน และดึงข้อมูลเที่ยวบินได้


## การติดตั้ง


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## การกำหนดเครื่องมือด้วยตัวตกแต่ง @tool

ตัวตกแต่ง `@tool` จะเปลี่ยนฟังก์ชัน Python ธรรมดาให้กลายเป็นเครื่องมือที่เอเจนต์สามารถเรียกใช้ได้
ประเด็นสำคัญ:

- **docstring** จะกลายเป็นคำอธิบายเครื่องมือที่โมเดลเห็น
- **คำอธิบายประเภท** (รวมถึง `Annotated` พร้อมคำอธิบาย) กำหนดโครงสร้างของเครื่องมือ
- `approval_mode` ควบคุมว่าผู้ใช้ต้องอนุมัติแต่ละครั้งก่อนเรียกใช้หรือไม่


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## การสร้างเอเจนต์ด้วยเครื่องมือหลายตัว

ส่งเครื่องมือทั้งสามไปยังไคลเอ็นต์เพื่อให้โมเดลสามารถเรียกใช้เครื่องมือที่ต้องการเพื่อตอบคำถามของผู้ใช้ได้ตามต้องการ


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Structured Output with Tools

โดยการตั้งค่า `response_format` เป็นโมเดล Pydantic ตัวแทนจะถูกบังคับให้ส่งกลับวัตถุ JSON ที่มีประเภทชัดเจนแทนข้อความแบบอิสระ ซึ่งมีประโยชน์เมื่อโค้ดที่ทำงานต่อจากนี้ต้องการใช้ผลลัพธ์ในรูปแบบโปรแกรมเมติก


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## รูปแบบการอนุมัติเครื่องมือ

พารามิเตอร์ `approval_mode` บน `@tool` ควบคุมว่าเครื่องมือจะต้องได้รับการอนุมัติจากมนุษย์ก่อนดำเนินการหรือไม่:

| โหมด | พฤติกรรม |
|---|---|
| `"never_require"` | เครื่องมือทำงานโดยอัตโนมัติ — ไม่ต้องการการยืนยันจากผู้ใช้ |
| `"always_require"` | ทุกคำสั่งต้องได้รับการอนุมัติจากผู้ใช้ก่อนดำเนินการ |

ใช้ `"always_require"` กับเครื่องมือที่มีผลข้างเคียง (เช่น การจองตั๋วเครื่องบิน, การชำระเงินด้วยบัตรเครดิต) เพื่อให้มนุษย์ยังคงควบคุมอยู่ในวงจร


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการ:

1. **กำหนดเครื่องมือ** โดยใช้ `@tool` decorator พร้อมกับพารามิเตอร์แบบ typed และ docstrings ที่ทำหน้าที่เป็น schema ของเครื่องมือ
2. **ประกอบเครื่องมือหลายตัว** เพื่อให้ agent สามารถเรียกใช้งานตามลำดับเพื่อให้ตอบคำถามที่ซับซ้อนได้
3. **ส่งคืนผลลัพธ์ที่มีโครงสร้าง** โดยการส่งผ่านโมเดล Pydantic เป็น `response_format`
4. **ควบคุมการอนุมัติเครื่องมือ** ด้วย `approval_mode` เพื่อให้มนุษย์มีส่วนร่วมในกรณีที่เกี่ยวข้องกับการทำงานที่ละเอียดอ่อน

รูปแบบเหล่านี้เป็นพื้นฐานในการสร้าง agents ที่น่าเชื่อถือและพร้อมสำหรับใช้งานจริงที่สามารถทำงานร่วมกับระบบภายนอกได้อย่างปลอดภัย


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
